<a href="https://colab.research.google.com/github/uiuxkelompok/uts-bigdata/blob/main/Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'com.supercell.clashofclans',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install transformers

In [ ]:
import pandas as pd
from transformers import pipeline

# Convert the list of dictionaries to a pandas DataFrame
df = pd.DataFrame(result)

# Initialize the sentiment analysis pipeline with the specified model
sentiment_analyzer = pipeline("sentiment-analysis", model="w11wo/indonesian-roberta-base-prdect-id")

# Perform sentiment analysis on the 'content' column
df['sentiment'] = df['content'].apply(lambda x: sentiment_analyzer(x)[0]['label'])
df['sentiment_score'] = df['content'].apply(lambda x: sentiment_analyzer(x)[0]['score'])

# Display the DataFrame with sentiment analysis results
display(df[['content', 'sentiment', 'sentiment_score']].head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style for the plots
sns.set_style("whitegrid")

# Create a figure with two subplots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Bar Chart of Sentiment Distribution
sentiment_counts = df['sentiment'].value_counts().sort_index()
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, ax=axes[0], palette='viridis')
axes[0].set_title('Distribusi Sentimen Ulasan Pengguna', fontsize=14)
axes[0].set_xlabel('Sentimen', fontsize=12)
axes[0].set_ylabel('Jumlah Ulasan', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# Add count labels on top of the bars
for index, value in enumerate(sentiment_counts.values):
    axes[0].text(index, value + 0.5, str(value), ha='center', va='bottom')

# Subplot 2: Pie Chart of Sentiment Proportion
colors = sns.color_palette('viridis', len(sentiment_counts))
axes[1].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%', startangle=90, colors=colors)
axes[1].set_title('Proporsi Sentimen Ulasan Pengguna', fontsize=14)
axes[1].axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.

plt.tight_layout()
plt.show()